# Single-drug FAERS: bagging and boosting comparison

This compares a bootstrap Random Forest and XGBoost with Morgan ExtraTrees using the same Morgan fingerprints. Five-fold scaffold-grouped validation on 2025 labels selects settings. The 2026 H1 comparison is exploratory because those results have already been reviewed. The target is a FAERS serious-outcome report fraction, not clinical toxicity risk.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, sys

DATA_CANDIDATES = [Path("/content/gdrive_auditddi_toxicity/MyDrive/Projects/auditddi"), Path("/content/gdrive_auditddi/MyDrive/Projects/auditddi"), Path("/content/drive/MyDrive/Projects/auditddi")]
DATA = next((p for p in DATA_CANDIDATES if p.is_dir()), None)
if DATA is None:
    from google.colab import drive
    MOUNT = Path("/content/gdrive_auditddi_trees")
    drive.mount(str(MOUNT))
    DATA = MOUNT / "MyDrive/Projects/auditddi"
assert DATA.is_dir(), f"Drive data root not found: {DATA}"

REPO = Path("/content/auditddi_toxicity_tree_ensembles")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "main", "https://github.com/akhilau-git/auditddi.git", str(REPO)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
    assert not dirty, "Code checkout has local changes. Use a fresh Colab runtime or clean checkout before pulling."
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("Drive data root:", DATA)
print("Code commit:", commit)

In [ ]:
%pip install -q "xgboost>=2.1,<4"
print("XGBoost installation complete.")

In [ ]:
import py_compile
SCRIPT = REPO / "scripts/compare_single_drug_tree_ensembles.py"
py_compile.compile(str(SCRIPT), doraise=True)
LABEL_ROOT = DATA / "results/pillars/single_drug_toxicity/label_builds"
label_dirs = sorted(p for p in LABEL_ROOT.glob("faers_2025_train_2026h1_test_*") if (p / "train_2025_suspect_roles.csv").is_file() and (p / "test_new_cases_2026h1_suspect_roles.csv").is_file())
assert label_dirs, f"Completed label files not found under {LABEL_ROOT}"
LABEL_DIR = label_dirs[-1]
baseline_dirs = sorted(p for p in (DATA / "results/pillars/single_drug_toxicity/temporal_models").glob("*five_seed_*") if (p / "temporal_predictions.csv").is_file())
assert baseline_dirs, "Five-seed ExtraTrees predictions not found. Complete that comparison first."
BASELINE_PREDICTIONS = baseline_dirs[-1] / "temporal_predictions.csv"
OUT = DATA / "results/pillars/single_drug_toxicity/tree_ensemble_comparisons" / f"ecfp6_scaffold_cv_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=False)
command = [sys.executable, "-u", str(SCRIPT), "--train-labels", str(LABEL_DIR / "train_2025_suspect_roles.csv"), "--test-labels", str(LABEL_DIR / "test_new_cases_2026h1_suspect_roles.csv"), "--output-dir", str(OUT), "--benchmark-predictions", str(BASELINE_PREDICTIONS)]
log_path = OUT / "tree_ensemble_comparison.log"
print("Reusing labels:", LABEL_DIR)
print("ExtraTrees reference:", BASELINE_PREDICTIONS)
print("Output:", OUT)
print("Live log:", log_path)
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process, log_path.open("w", encoding="utf-8") as log:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    return_code = process.wait()
if return_code != 0: raise RuntimeError(f"Comparison failed with exit code {return_code}. See {log_path}")
print("Per-seed metrics:", OUT / "tree_ensemble_temporal_metrics.csv")
print("Seed summary:", OUT / "tree_ensemble_seed_summary.csv")
print("2025 validation search:", OUT / "scaffold_cv_model_search.csv")